# Wildfire HTTP Evaluation + Final Artifacts

Use this notebook after GRPO training. All **code** for this pipeline lives here
and in `wildfire_training_eval_hf.ipynb` (no separate one-off scripts required
in the notebook flow).

**Comparing the two “LLMs” (base Qwen vs LoRA adapter):** you get that from
**`eval_untrained.json` and `eval_trained.json`**, which use the *same* held-out
seeds per task, so every score pair is a fair A/B.  
**`grpo_wildfire/log.jsonl`** is different: it is one model’s **training** run
over iterations (not a two-policy comparison), though `--plot-training` can
export training curves for the same submission folder.

Run order (top to bottom): **1 → 2 → 3 → 4 → 5 → 6 → 7**.

1. Pull repo, install `matplotlib`, set `BASE` / `SPACE_URL`.
2. Untrained OpenEnv eval → `eval_untrained.json`.
3. Trained eval; add `--plot-training` to also write `training_*.png/.svg` + `training_summary.md` from **`grpo_wildfire/log.jsonl`** (same as `plot_training_curves.py`). Eval HTTP results do *not* draw these; the **training log** does.
4. Base vs LoRA comparison charts (from both eval JSONs).
5. **Inline display** of the training-curve PNGs + `training_summary.md` in the notebook (if present under `submission_artifacts/`).
6. Optional: upload `grpo_wildfire/` to a HF model repo.
7. Print `overall_mean` from the eval JSONs.


In [ ]:
# Cell 1: repo update + constants
import os
import subprocess
import sys

BASE = "/home/user/app/wildfire-env"
SPACE_URL = "https://chunchunmaru-101-wildfire-env.hf.space"
os.chdir(BASE)

subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
os.makedirs("submission_artifacts", exist_ok=True)
print("cwd:", os.getcwd())
print("OpenEnv HTTP base:", SPACE_URL)


In [ ]:
# Cell 2: untrained baseline eval (OpenEnv WebSocket + /grader)
import subprocess
import sys

subprocess.run([
    sys.executable,
    "eval_policy_http.py",
    "--untrained",
    "--base-url",
    SPACE_URL,
    "--seeds-per-task",
    "5",
    "--output",
    "submission_artifacts/eval_untrained.json",
], check=True)
print("Wrote OpenEnv baseline: submission_artifacts/eval_untrained.json")


In [ ]:
# Cell 3: trained eval + training artifacts (PNG/SVG/summary from grpo_wildfire/log.jsonl)
import subprocess
import sys

subprocess.run([
    sys.executable,
    "eval_policy_http.py",
    "--base-url",
    SPACE_URL,
    "--seeds-per-task",
    "5",
    "--output",
    "submission_artifacts/eval_trained.json",
    "--plot-training",
], check=True)
print("Wrote eval_trained.json + submission_artifacts/training_* if log.jsonl was present")


In [ ]:
# Cell 4: base (untrained) vs fine-tuned (adapter) — needs eval JSONs from Cells 2 & 3
import json
from pathlib import Path
import matplotlib.pyplot as plt

_un = Path("submission_artifacts/eval_untrained.json")
_tr = Path("submission_artifacts/eval_trained.json")
if not (_un.is_file() and _tr.is_file()):
    raise FileNotFoundError("Run Cells 2 and 3 first so both eval JSONs exist.")
with _un.open(encoding="utf-8") as f:
    du = json.load(f)
with _tr.open(encoding="utf-8") as f:
    dt = json.load(f)

tasks = [t for t in ("easy", "medium", "hard") if t in du and t in dt]


def _by_seed(block: dict) -> dict[int, float]:
    return {int(ep["seed"]): float(ep["score"]) for ep in block["episodes"]}


# --- Table: per-task + overall
print(f"Base model: {du.get('model_name')}  |  trained adapter: {dt.get('adapter_path', 'see JSON')}")
print(f"Overall mean /grader  untrained={du.get('overall_mean'):.4f}  trained={dt.get('overall_mean'):.4f}  "
      f"Δ={float(dt['overall_mean']) - float(du['overall_mean']):+.4f}\n")
print("| task | n | untrained mean | trained mean | Δ |")
print("|---|---:|---:|---:|---:|")
for t in tasks:
    u_m, tr_m = float(du[t]["mean"]), float(dt[t]["mean"])
    n = len(du[t]["episodes"])
    print(f"| {t} | {n} | {u_m:.4f} | {tr_m:.4f} | {tr_m - u_m:+.4f} |")

# --- Bar: per-task means
u_means = [float(du[t]["mean"]) for t in tasks]
t_means = [float(dt[t]["mean"]) for t in tasks]
x = list(range(len(tasks)))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([i - w / 2 for i in x], u_means, w, label="untrained (base)", color="tab:blue")
ax.bar([i + w / 2 for i in x], t_means, w, label="trained (LoRA)", color="tab:orange")
ax.set_xticks(x, tasks)
ax.set_ylabel("Mean /grader score")
ax.set_title("OpenEnv showcase — same held-out seeds per task")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# --- Scatter: paired (x=untrained, y=trained), one point per episode; y=x reference
fig, ax = plt.subplots(figsize=(5.5, 5.5))
colors = {"easy": "tab:green", "medium": "tab:purple", "hard": "tab:red"}
for t in tasks:
    su, st = _by_seed(du[t]), _by_seed(dt[t])
    for seed in sorted(su.keys() & st.keys()):
        ax.scatter(su[seed], st[seed], c=colors.get(t, "gray"), s=45, alpha=0.85, edgecolors="white", linewidths=0.5)
    # task label once: single proxy point
    ax.scatter([], [], c=colors.get(t, "gray"), s=45, label=t)
ax.plot([0, 1], [0, 1], "k--", alpha=0.35, linewidth=1)
ax.set_xlabel("untrained /grader")
ax.set_ylabel("trained /grader")
ax.set_title("Paired scores (same seed = same episode layout)")
ax.legend(title="task")
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_box_aspect(1)
plt.tight_layout()
plt.show()

# --- Per task: lines over seeds (paired)
fig, axes = plt.subplots(1, len(tasks), figsize=(12, 3.5), sharey=True)
for ax, t in zip(axes, tasks):
    su, st = _by_seed(du[t]), _by_seed(dt[t])
    seeds = sorted(su.keys() & st.keys())
    idx = list(range(len(seeds)))
    ax.plot(idx, [su[s] for s in seeds], "o-", label="base", color="tab:blue")
    ax.plot(idx, [st[s] for s in seeds], "s-", label="LoRA", color="tab:orange")
    ax.set_xticks(idx, [str(s) for s in seeds], rotation=45, fontsize=8)
    ax.set_xlabel("seed")
    ax.set_title(t)
    ax.set_ylim(0, 1.05)
axes[0].set_ylabel("/grader")
axes[0].legend(loc="lower right", fontsize=8)
plt.suptitle("Per-seed paired comparison", y=1.02)
plt.tight_layout()
plt.show()

# --- Histogram: trained - untrained (15 episodes total for 5×3 tasks)
deltas = []
for t in tasks:
    su, st = _by_seed(du[t]), _by_seed(dt[t])
    for seed in su.keys() & st.keys():
        deltas.append(st[seed] - su[seed])
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(deltas, bins=8, color="steelblue", edgecolor="white")
ax.axvline(0, color="k", linestyle="--", alpha=0.5)
ax.set_xlabel("trained − untrained /grader")
ax.set_title(f"Δ per episode (n={len(deltas)})")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 5: show training-curve files on disk (from `--plot-training` / `plot_training_curves.py`)
#
# Data source: `grpo_wildfire/log.jsonl` (training iters) — not the OpenEnv eval JSONs.
from pathlib import Path
from IPython.display import Image, display, Markdown

_arts = Path("submission_artifacts")
for fname in ("training_reward_curve.png", "training_loss_curve.png"):
    p = _arts / fname
    if p.is_file():
        display(Markdown(f"### `{p}`"))
        display(Image(filename=str(p)))
    else:
        print("Missing (train first + run trained eval with --plot-training on a machine that has the log):", p)

_s = _arts / "training_summary.md"
if _s.is_file():
    display(Markdown("### `training_summary.md`"))
    display(Markdown(_s.read_text(encoding="utf-8")))
else:
    print("No training_summary.md at", _s)

In [ ]:
# Cell 6: upload checkpoints to HF model repo (optional but recommended)
import os
from huggingface_hub import HfApi, create_repo

HF_REPO_ID = os.environ.get("HF_REPO_ID", "Chunchunmaru-101/wildfire-grpo-checkpoints")
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN before upload.")

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO_ID, repo_type="model", exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path="grpo_wildfire",
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload wildfire GRPO checkpoints",
    ignore_patterns=["*.pyc", "__pycache__/*"],
)
print(f"Uploaded grpo_wildfire to https://huggingface.co/{HF_REPO_ID}")


In [ ]:
# Cell 7: quick artifact view
import json

for path in ["submission_artifacts/eval_untrained.json", "submission_artifacts/eval_trained.json"]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(path, "overall_mean=", data.get("overall_mean"))
